In [102]:
import os
import sys
import re
import spacy
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from dateutil import parser
from Levenshtein import ratio
from PIL import Image

In [103]:
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'transcription')))

In [ ]:
def open_textfile(image_path: str) -> dict:
    # build ground truth dictionaries 
    data_dict = {}
    with open(image_path, "r") as f:
        lines = f.readlines()
        for line in lines:
            try:
                pattern = r"(?<=\d):"  # Match ':' only if preceded by a digit
                key, value = re.split(pattern, line.strip())
                data_dict[key] = value
            except Exception as e:
                return f"Error reading line in file: {line}: {e}" 
    return data_dict

In [105]:
# focus on evaluating taxon, geography, and date 

# event dates ground truth labels
dates_path = "../transcription/data/new-england-samples/output/dates.txt"
dates_dict = open_textfile(dates_path)

# geography ground truth labels 
geography_path = "../transcription/data/new-england-samples/output/localities.txt"
geography_dict = open_textfile(geography_path)

# taxon ground truth labels
taxon_path = "../transcription/data/new-england-samples/output/taxons.txt"
taxon_dict = open_textfile(taxon_path)

In [ ]:
results = pd.read_csv("../transcription/results/tesseract_results.csv", index_col=False)

In [ ]:
# to do: for extracting taxon name try using a specific ner package instead this is a workaround for now 
def extract_binomial_names(text: str) -> list:
    # matches "Genus species" format (capitalized genus, lowercase species)
    pattern = r'\b([A-Z][a-z]+)\s([a-z]{2,})\b'
    matches = re.findall(pattern, text)

    return [' '.join(match) for match in matches]

# try to fuzzy match extracted text - taxon to gbif 
def validate_plant_name_gbif(name: str):
    url = f"https://api.gbif.org/v2/species/match?scientificName={name}" # fuzzy matching
    response = requests.get(url)
    if response.status_code != 200:
        return None

    data = response.json()
    
    if data["diagnostics"]["matchType"] == "EXACT":
        return {
            "valid_name": data["usage"]["canonicalName"],
            "status": data["usage"]["status"],
            "confidence": data["diagnostics"]["confidence"]
        }
    return None

# parse extracted text for taxon name, location, and date collected and return candidates for each image
def process_text(combined_text: str, nlp) -> dict:
    doc = nlp(combined_text)

    results = {}
    taxon_name = []
    locations = []
    dates = []

    # save location (GPE) and date entities (DATE)
    labels_of_interest = ["GPE", "DATE"]
    for ent in doc.ents:
        if ent.label_ in labels_of_interest:
            if ent.label_ == "DATE":
                dates.append(ent.text)
            else:
                locations.append(ent.text)
    
    # search for taxon name 
    # step 1: extract potential names
    candidates = extract_binomial_names(combined_text)

    # step 2: validate
    taxon_name = []
    for name in candidates:
        result = validate_plant_name_gbif(name)
        if result:
            taxon_name.append(result['valid_name']) # only append matched taxon name
        else:
            taxon_name.append(name)

    # package ocr results 
    results["scientificName"] = taxon_name
    results["location"] = locations
    results["eventDate"] = dates

    return results

notes:
- text blob extract from tesseract 
  - spacy ner extracts
    - date 
    - location
  - regex for taxon name 
    - gbif validation
      - yes - thats the result 
      - no - return original extracted name 

- after processing, extracted text has multiple options 
  - for each item in list 
    - compare item with ground truth get levenshtein distance and save score 
    - highest score is the predicted value 

In [78]:
def get_best_candidate(predicted_values: list, actual_value):
    if predicted_values:
        # compute levenshtein ratio between the actual value and each value in the predicted values list 
        final_pred_value, value_ratio_max = max([(pred_v, ratio(actual_value.lower(), pred_v.lower())) for pred_v in predicted_values], 
                                                  key=lambda x: x[1])
        return final_pred_value, value_ratio_max
    
    return predicted_values, 0  

In [86]:
def append_results(results_df: pd.DataFrame) -> pd.DataFrame:
    # initialize the necessary columns in the dataframe
    results_df["Actual Taxon"] = ""
    results_df["Predicted Taxon"] = ""
    results_df["Lev Ratio Taxon"] = 0

    results_df["Actual Geography"] = ""
    results_df["Predicted Geo"] = ""
    results_df["Lev Ratio Geo"] = 0

    results_df["Actual Date"] = ""
    results_df["Predicted Date"] = ""
    results_df["Lev Ratio Date"] = 0

    # load spacy NER 
    nlp = spacy.load("en_core_web_sm")

    for idx, row in results_df.iterrows():
        processed_text_results = process_text(row["combined_text"], nlp)

        # get predicted values
        predicted_taxons = processed_text_results.get("scientificName", [])
        predicted_geographies = processed_text_results.get("location", [])
        predicted_dates = processed_text_results.get("eventDate", [])

        # parse image path for occurrence id 
        occid = str(row["image_path"].split("/")[-1].split(".jpeg")[0])

        # get actual 
        actual_taxon = taxon_dict.get(occid, "")
        actual_geography = geography_dict.get(occid, "")
        actual_date = dates_dict.get(occid, "")

        # append actual values to dataframe columns
        results_df.at[idx, "Actual Taxon"] = actual_taxon
        results_df.at[idx, "Actual Geography"] = actual_geography
        results_df.at[idx, "Actual Date"] = actual_date

        # get the best candidate for each category (taxon, location, date)
        final_pred_taxon, taxon_ratio_max = get_best_candidate(predicted_taxons, actual_taxon) 
        final_pred_geo, geo_ratio_max = get_best_candidate(predicted_geographies, actual_geography)
        final_pred_date, date_ratio_max = get_best_candidate(predicted_dates, actual_date)

        # append predicted values and levenshtein ratios dataframe columns 
        results_df.at[idx, "Predicted Taxon"] = final_pred_taxon
        results_df.at[idx, "Lev Ratio Taxon"] = taxon_ratio_max
        results_df.at[idx, "Predicted Geo"] = final_pred_geo
        results_df.at[idx, "Lev Ratio Geo"] = geo_ratio_max
        results_df.at[idx, "Predicted Date"] = final_pred_date
        results_df.at[idx, "Lev Ratio Date"] = date_ratio_max

    return results_df

In [87]:
predictions_df = append_results(results)

/var/folders/2y/y5t2qnws7q90nyp0j53qcbtc00cbkp/T/ipykernel_45479/3760568043.py:46: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.2068965517241379' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  results_df.at[idx, "Lev Ratio Taxon"] = taxon_ratio_max
/var/folders/2y/y5t2qnws7q90nyp0j53qcbtc00cbkp/T/ipykernel_45479/3760568043.py:48: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.6976744186046512' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  results_df.at[idx, "Lev Ratio Geo"] = geo_ratio_max
/var/folders/2y/y5t2qnws7q90nyp0j53qcbtc00cbkp/T/ipykernel_45479/3760568043.py:50: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5333333333333333' has dtype incompat

In [ ]:
# optional write results to csv 
predictions_df.to_csv("/Users/mvoong/Desktop/spark-symbiota-ml/transcription/results/tesseract_results_appended.csv")

In [ ]:
def calculate_accuracy(results_df: pd.DataFrame, column_name: str, sim_threshold=0.8) -> float:
    acc_count = (results_df[column_name] > sim_threshold).sum()    
    acc_val = acc_count/len(results_df)
    return acc_val*100, acc_count

In [94]:
def review_images(image_paths: list):
    # open and display images 
    for img_path in image_paths:
        if os.path.exists(img_path): # check if the path exists 
            try:
                img = Image.open(img_path)
                img.show()
            except Exception as e:
                print(f"Error opening image {img_path}: {e}")
        else:
            print(f"Path does not exist: {img_path}")

In [ ]:
# thresholds 
# 0.8 > taxon -- needs to be relatively high to have gotten most of the name 
# 0.6 > geography -- could have gotten part of the location name 
# 0.5 > date -- could have gotten part of the date, some combination of year, day, month 

taxon_acc, taxon_count = calculate_accuracy(predictions_df, "Lev Ratio Taxon", 0.8)
geo_acc, geo_count = calculate_accuracy(predictions_df, "Lev Ratio Geo", 0.6)
date_acc, date_count = calculate_accuracy(predictions_df, "Lev Ratio date", 0.5)

In [ ]:
print(taxon_acc)
predictions_df[predictions_df["Lev Ratio Taxon"] > 0.8]

12.0


,Unnamed: 0,image_path,combined_text,avg_conf,Actual Taxon,Predicted Taxon,Lev Ratio Taxon,Actual Geography,Lev Ratio Geo,Predicted Geo,Actual Date,Predicted Date,Lev Ratio date,Lev Ratio Date
11,11,1228621608.jpeg,"jo 2 ova oo 7s 0 10 St MheNel e, copy...",76.164997,Juncus bufonius,Juncus bufonius,0.967742,Nantasket Beach,0.190476,Mass.,1889-06-23,1892,0.533333,0.533333
21,21,1038980442.jpeg,HERBAR iy YALE UN ae erty ean OLIN YU.058217 H...,68.201809,Chimaphila umbellata,Chimaphila umbellate,0.926829,tract south of Maltby Lakes,0.125000,Conn,1912-04-25,25 1912,0.555556,0.555556
29,29,5077516702.jpeg,III UAE V 0338003 WI S} Proserpinaca pectinata...,73.335864,Proserpinaca pectinata,Proserpinaca pectinata,0.977778,"Grassy Pond, Hopkinton",0.562500,Hopkinton,1914-09-03,1791,0.400000,0.400000
33,33,5102793515.jpeg,~ WASHINGTON ...,67.249393,Nabalus altissima,Nabalus altissimus,0.888889,Vicinity of Wolfeboro,0.250000,WASHINGTON,1909,February 2025,0.222222,0.222222
43,43,1262190966.jpeg,10 copyright reserved The Field Museum (F ‘WMI...,74.198639,Lupinus perennis,Lupinus perennis,0.969697,Needham,0.133333,Lupinus,1884-06-07,219204,0.235294,0.235294
44,44,1038964757.jpeg,PLANTS OF CONNECTICUT No. 47 Family: Gramineae...,72.347621,Avena sativa,Avena sativa,0.960000,"Meadow Street, small cinder pile in vacant lo...",0.131868,Branford,1877-06,[],0.000000,0.000000


In [ ]:
# review the images, it's ok because there are not that many :'(
base_path = "../transcription/data/new-england-samples/output/"
img_path_high_taxon_acc = base_path + predictions_df[predictions_df["Lev Ratio Taxon"] > 0.8]["image_path"]
review_images(img_path_high_taxon_acc)

In [91]:
print(geo_acc)
predictions_df[predictions_df["Lev Ratio Geo"] > 0.6]

2.0


,Unnamed: 0,image_path,combined_text,avg_conf,Actual Taxon,Predicted Taxon,Lev Ratio Taxon,Actual Geography,Lev Ratio Geo,Predicted Geo,Actual Date,Predicted Date,Lev Ratio date,Lev Ratio Date
2,2,1038983765.jpeg,copyright reserved sii YU.008327 FLORA OF MONH...,66.116646,Osmundastrum cinnamomeum,Oswwmcdla enmeanmewna,0.434783,"Monhegan Island, Burnt Head",0.697674,MONHEGAN ISLAND,1919-07-01,1919,0.533333,0.533333


In [100]:
img_path_high_geo_acc = base_path + predictions_df[predictions_df["Lev Ratio Geo"] > 0.6]["image_path"]
review_images(img_path_high_geo_acc)

In [92]:
print(date_acc)
predictions_df[predictions_df["Lev Ratio date"] > 0.5]

10.0


,Unnamed: 0,image_path,combined_text,avg_conf,Actual Taxon,Predicted Taxon,Lev Ratio Taxon,Actual Geography,Lev Ratio Geo,Predicted Geo,Actual Date,Predicted Date,Lev Ratio date,Lev Ratio Date
2,2,1038983765.jpeg,copyright reserved sii YU.008327 FLORA OF MONH...,66.116646,Osmundastrum cinnamomeum,Oswwmcdla enmeanmewna,0.434783,"Monhegan Island, Burnt Head",0.697674,MONHEGAN ISLAND,1919-07-01,1919,0.533333,0.533333
5,5,1995316443.jpeg,: /-61G Universit: - of Sc ith Florida terbari...,61.987847,Salix sericea,Hampshire rey,0.370370,Peterborough.,0.222222,New Hampshire,1907-07-29,29 1907,0.555556,0.555556
11,11,1228621608.jpeg,"jo 2 ova oo 7s 0 10 St MheNel e, copy...",76.164997,Juncus bufonius,Juncus bufonius,0.967742,Nantasket Beach,0.190476,Mass.,1889-06-23,1892,0.533333,0.533333
21,21,1038980442.jpeg,HERBAR iy YALE UN ae erty ean OLIN YU.058217 H...,68.201809,Chimaphila umbellata,Chimaphila umbellate,0.926829,tract south of Maltby Lakes,0.125000,Conn,1912-04-25,25 1912,0.555556,0.555556
37,37,1038919124.jpeg,3 3 4 5 3 2 = 2 | Q S 8 cticut Bot HU CBS.0134...,64.640406,Asplenium trichomanes,Shaded ledges,0.285714,Shaded ledges,0.363636,Hartland,1914-06-21,1914,0.533333,0.533333


In [101]:
img_path_high_date_acc = base_path + predictions_df[predictions_df["Lev Ratio date"] > 0.5]["image_path"]
review_images(img_path_high_date_acc)